# Phosphorylation site map - September 6, 2015
Map modified phosphorylation sites for peptides in all three replicates

## Configuration

In [1]:
import pandas as pd
import os

data_dir = os.path.join('..','..','phosphorylation-data-sets','McKenna2014')

## Load all files

In [2]:
# Use all three replicates
rep_a_df = pd.read_csv(os.path.join(data_dir,'simplified_biological_replicate_A_orig.txt'), sep='\t')
rep_b_df = pd.read_csv(os.path.join(data_dir,'simplified_biological_replicate_B_orig.txt'), sep='\t')
rep_c_df = pd.read_csv(os.path.join(data_dir,'simplified_biological_replicate_C_orig.txt'), sep='\t')

# Create dictionaries of peptide to modified site for each replicate
rep_a_dict = dict(zip(rep_a_df['peptide'], rep_a_df['ModifiedSites']))
rep_b_dict = dict(zip(rep_b_df['peptide'], rep_b_df['ModifiedSites']))
rep_c_dict = dict(zip(rep_c_df['peptide'], rep_c_df['ModifiedSites']))

# The list of peptides in the intersection, sorted in the desired order
agg_df = pd.read_csv(os.path.join(data_dir,'aggregated_replicateABC_tukey_orig_firstprev.txt'), sep='\t')
print 'Mapping sites for %d peptides' % len(agg_df['peptide'])

Mapping sites for 1068 peptides


## Write the output mapping
When replicates contain different sites for the same peptide, take the union of the reported sites

In [19]:
with open(os.path.join(data_dir, 'peptide_site_map_090615.txt'), 'w') as writer:
    writer.write('peptide\tmodified.sites\n')
    
    for peptide in agg_df['peptide']:
        site_a = rep_a_dict[peptide]
        site_b = rep_b_dict[peptide]
        site_c = rep_c_dict[peptide]
        # Check that the other replicates agree with the site
        if site_b != site_a or site_c != site_a:
            # Report the disagreement
            print 'Replicates list different sites for %s' % peptide
            print '%s\n%s\n%s' % (site_a, site_b, site_c)
            # Take the union of all sites reported.  Because the only instances
            # involve tyrosine phosphorylation we can sort the strings instead
            # of sorting the numeric index of the site.  Don't need to handle double
            # quotes.
            all_sites = set([site_a, site_b, site_c])
            all_sites = sorted(list(all_sites))
            all_sites = ','.join(all_sites)
            print 'Unique sites: %s\n' % all_sites
            writer.write('%s\t%s\n' % (peptide, all_sites))
        else: # Simple case, all replicates agree on the site
            # Remove the double quotes when there are multiple sites
            writer.write('%s\t%s\n' % (peptide, site_a.replace('"', '')))

Replicates list different sites for K.n[305.21]AVC[160.03]STY[243.03]LQSR.Y
Y361
Y352
Y361
Unique sites: Y352,Y361

Replicates list different sites for R.n[305.21]IYQY[243.03]IQSR.F
Y273
Y321
Y273
Unique sites: Y273,Y321

